Transformer 模型完全基于注意力机制，没有卷积层或循环神经网络层
Transformer 的编码器和解码器由基于自注意力的模块叠加而成

Transformer 的每个层都有两个子层：多头自注意力 + 基于位置的前馈网络，并在层中使用残差链接和层规范化

In [ ]:
"准备数据"

import re
import random

RAW_TSV = "Tatoeba-eng-fra-train.tsv"
DEV_TSV = "Tatoeba-eng-fra-dev.tsv"

# 过滤参数
MIN_WORD = 3
MAX_WORD = 40
RATIO_MIN = 0.5
RATIO_MAX = 2.0

# 输出训练验证文件
TRAIN_SRC = "train_src.txt"
TRAIN_TGT = "train_tgt.txt"
VAL_SRC = "val_src.txt"
VAL_TGT = "val_tgt.txt"

# 文本清洗
def clean_text(s: str) -> str:
    s = re.sub(r"(http|https)://\S+|@\w+|#\w+", "", s)
    s = re.sub(r"\s+", " ", s.strip())
    return s.lower()

def word_count(s: str) -> int:
    return len(s.split())

def load_tsv(file_path):
    pairs = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")

            if len(parts) < 3:
                continue
            en_text = parts[1]
            fr_text = parts[2]
            en_clean = clean_text(en_text)
            fr_clean = clean_text(fr_text)
            pairs.append((en_clean, fr_clean))
    return pairs

# 加载数据
train_raw = load_tsv(RAW_TSV)
dev_raw = load_tsv(DEV_TSV)
all_raw_pairs = train_raw + dev_raw

# 过滤低质量句子
valid_pairs = []
for en, fr in all_raw_pairs:
    if len(en) < 2 or len(fr) < 2:
        continue
    en_word_num = word_count(en)
    fr_word_num = word_count(fr)
    if not (MIN_WORD <= en_word_num <= MAX_WORD and MIN_WORD <= fr_word_num <= MAX_WORD):
        continue
    length_ratio = en_word_num / fr_word_num
    if not (RATIO_MIN <= length_ratio <= RATIO_MAX):
        continue
    valid_pairs.append((en, fr))


# 打乱切分训练/验证
random.seed(42)
random.shuffle(valid_pairs)
split_index = int(len(valid_pairs) * 0.9)
train_dataset = valid_pairs[:split_index]
val_dataset = valid_pairs[split_index:]

# 保存txt文件
def save_pair_file(pair_list, src_out, tgt_out):
    with open(src_out, "w", encoding="utf-8") as f_src, open(tgt_out, "w", encoding="utf-8") as f_tgt:
        for en_sent, fr_sent in pair_list:
            f_src.write(en_sent + "\n")
            f_tgt.write(fr_sent + "\n")

save_pair_file(train_dataset, TRAIN_SRC, TRAIN_TGT)
save_pair_file(val_dataset, VAL_SRC, VAL_TGT)

In [ ]:
"Load Data"

import re
import collections
from collections import Counter
from torch.utils.data import Dataset, DataLoader

def read_text_file(file_path: str) -> list[str]:

    # 打开文件：指定utf-8编码，遇异常字符自动替换，避免乱码报错
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    # 逐行清洗，额外过滤空行
    cleaned_lines = []
    for line in lines:
        # 只把连续空白替换为单个空格，保留字母、' . !
        line_processed = re.sub('\s+', ' ', line).strip().lower()
        # 过滤空行
        if line_processed:
            cleaned_lines.append(line_processed)

    return cleaned_lines

def tokenize(lines, token='word'):
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('unknowed token:' + token)

def count_corpus(tokens):
    "统计语料词频"

    # 空输入直接返回空计数器
    if len(tokens) == 0:
        return Counter()
    # 判断是二维嵌套列表（每行分词结果），展平为一维
    if isinstance(tokens[0], list):
        tokens = [token for line in tokens for token in line]
    return Counter(tokens)


class Vocab:
    
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        # 空输入默认初始化为空列表
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []

        # 1. 统计全部词元频率，并按频率从高到低排序
        counter = count_corpus(tokens)
        # 按词频降序排列 (token, freq)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)

        # 2. 初始化索引-词双向映射
        # 首位固定<unk>未知词，再拼接用户自定义保留词
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}

        # 3. 过滤低频词，构建完整词表映射
        for token, freq in self._token_freqs:
            # 低于最小频率阈值直接跳过，后续全部不再遍历
            if freq < min_freq:
                break
            # 避免和保留token重复
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):

        # 单个词元输入
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        # 列表/元组批量转换
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[idx] for idx in indices]

    @property
    def unk(self):
        "未知词元<unk>固定索引为0"
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def truncate_pad(seq: list, max_len: int, pad_idx: int):
    """
    把序列截断 / 填充到固定长度，同时返回有效长度
    返回: padding后的序列, 有效长度
    """
    if len(seq) > max_len:
        return seq[:max_len], max_len
    valid_len = len(seq)
    padded = seq + [pad_idx] * (max_len - len(seq))
    return padded, valid_len

class NMTDataset(Dataset):
    def __init__(self, src_file: str, tgt_file: str, max_len: int, 
                 min_freq: int = 1, device=None):
        src_lines = read_text_file(src_file)
        tgt_lines = read_text_file(tgt_file)
        assert len(src_lines) == len(tgt_lines), "源文件和目标文件行数必须一一对应"
        src_tokens = tokenize(src_lines, 'word')
        tgt_tokens = tokenize(tgt_lines, 'word')

        self.src_vocab = Vocab(
            src_tokens, min_freq=min_freq,
            reserved_tokens=['<pad>', '<bos>', '<eos>']
        )
        self.tgt_vocab = Vocab(
            tgt_tokens, min_freq=min_freq,
            reserved_tokens=['<pad>', '<bos>', '<eos>']
        )

        src_seqs, tgt_seqs = [], []
        src_valid_lens, tgt_valid_lens = [], []
        for s_tok, t_tok in zip(src_tokens, tgt_tokens):
            s_idx = self.src_vocab[s_tok] + [self.src_vocab['<eos>']]
            t_idx = self.tgt_vocab[t_tok] + [self.tgt_vocab['<eos>']]
            s_pad, s_len = truncate_pad(s_idx, max_len, self.src_vocab['<pad>'])
            t_pad, t_len = truncate_pad(t_idx, max_len, self.tgt_vocab['<pad>'])
            src_seqs.append(s_pad)
            tgt_seqs.append(t_pad)
            src_valid_lens.append(s_len)
            tgt_valid_lens.append(t_len)

        # 转张量
        self.src_seqs = torch.tensor(src_seqs, dtype=torch.long)
        self.tgt_seqs = torch.tensor(tgt_seqs, dtype=torch.long)
        self.src_valid_lens = torch.tensor(src_valid_lens, dtype=torch.long)
        self.tgt_valid_lens = torch.tensor(tgt_valid_lens, dtype=torch.long)

        # 关键：指定设备则一次性全部搬到显存，训练零拷贝
        if device is not None:
            self.src_seqs = self.src_seqs.to(device)
            self.tgt_seqs = self.tgt_seqs.to(device)
            self.src_valid_lens = self.src_valid_lens.to(device)
            self.tgt_valid_lens = self.tgt_valid_lens.to(device)

    def __len__(self):
        return len(self.src_seqs)

    def __getitem__(self, idx):
        return (self.src_seqs[idx], self.tgt_seqs[idx],
                self.src_valid_lens[idx], self.tgt_valid_lens[idx])

def load_data_nmt(batch_size: int, max_len: int, 
                src_file: str, tgt_file: str,
                min_freq: int = 1, shuffle: bool = True,
                device=None):
    """
    加载机器翻译数据集
    返回: 数据迭代器, 源词表, 目标词表
    """
    dataset = NMTDataset(src_file, tgt_file, max_len, min_freq, device=device)
    train_iter = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )
    return train_iter, dataset.src_vocab, dataset.tgt_vocab

多头注意力只进行横向序列维度融合，无法改造单个 token 的内部特征（$V = W_V X$ 只是单层线性变换，无法把 X 映射到语义空间）

所以要使用 FFN，引入全局非线性，让模型能够拟合复杂语言（FFN 中含有非线性激活，理论上可以拟合任意连续语义函数）

In [ ]:
import math
import pandas as pd
import torch
from torch import nn

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

class PositionalEncoding(nn.Module):
    """
    正弦余弦固定位置编码（Transformer原文标准实现）
    为序列补充绝对位置信息，同时天然支持相对位置线性变换，无任何可学习参数。
    Args:
        d_model: 词嵌入/编码的特征维度，必须为偶数
        dropout: Dropout概率，默认0.1
        max_len: 预支持的最大序列长度，默认5000
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()

        # 工程强校验：维度必须为偶数，保证sin/cos成对对齐
        if d_model % 2 != 0:
            raise ValueError(f"d_model 必须为偶数，当前输入: {d_model}")

        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model

        # 预计算位置编码矩阵
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
        # 指数形式计算频率衰减项，数值更稳定（等价于 1/10000^(2i/d_model)）
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )

        # 初始化编码矩阵
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维 sin
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维 cos

        # 增加batch维度，注册为模型buffer（随模型自动迁移设备、保存到state_dict）
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: 输入词嵌入，形状 [batch_size, seq_len, d_model]
        Returns:
            叠加位置编码后的张量，形状与输入一致
        """
        seq_len = x.size(1)
        # 直接取对应长度的位置编码相加，buffer自动与输入同设备
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)

In [ ]:
class PositionWiseFFN(nn.Module):
    """基于位置的前馈网络"""
    def __init__(self, d_model: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 输入 → 线性1 → ReLU → 线性2 → 返回
        return self.net(x)

1. 残差连接
   解决深度网络梯度消失的问题
2. 层归一化
   稳定特征分布
   加速收敛
3. Dropout 正则化
   缓解过拟合

In [ ]:
class AddNorm(nn.Module):
    """残差连接后进行层规范化"""
    
    def __init__(self, normalized_shape, dropout, **kwargs):
        super(AddNorm, self).__init__(**kwargs)
        # Dropout：正则化，随机失活子层输出，防止过拟合
        self.dropout = nn.Dropout(dropout)
        # LayerNorm：传入特征维度大小，对每个token特征做归一化
        self.ln = nn.LayerNorm(normalized_shape)

    def forward(self, X, Y):
        # 1. 对子层输出Y做dropout
        # 2. 残差相加：原始输入X + 处理后的子层输出
        # 3. 对相加结果执行层归一化
        return self.ln(self.dropout(Y) + X)

EncoderBlock 类包含两个子层：多头字注意力 + 基于位置的前馈网络（FFN），且都使用残差连接 + 层规范化

In [ ]:
class TransformerEncoderLayer(nn.Module):
    """
    Transformer 编码器层
    结构：自注意力 -> AddNorm -> FFN -> AddNorm
    """

    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        # 多头自注意力
        self.self_attn = nn.MultiheadAttention(
            embed_dim = d_model,
            num_heads = nhead,
            dropout = dropout,
            batch_first=True  # 输入形状 [batch, seq_len, d_model]
        )
        # FFN + 两层AddNorm
        self.ffn = PositionWiseFFN(d_model, dim_feedforward, dropout)
        self.norm1 = AddNorm(d_model, dropout)
        self.norm2 = AddNorm(d_model, dropout)

    def forward(self, src: torch.Tensor, src_padding_mask: torch.Tensor = None):
        """
        Args:
            src: [batch, seq_len, d_model] 输入特征
            src_padding_mask: [batch, seq_len] 布尔掩码，True表示该位置是padding，注意力忽略
        Returns:
            out: [batch, seq_len, d_model] 输出特征
            attn_weights: [batch, nhead, seq_len, seq_len] 注意力权重
        """
        # 1. 多头自注意力 + 残差归一化
        attn_out, attn_weights = self.self_attn(
            query=src, key=src, value=src,
            key_padding_mask=src_padding_mask,
            need_weights=False
        )
        src = self.norm1(src, attn_out)

        # 2. 逐位置前馈网络 + 残差归一化
        ffn_out = self.ffn(src)
        src = self.norm2(src, ffn_out)

        return src, attn_weights

# ==================== Transformer 编码器 ====================
class TransformerEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 3,
        dim_feedforward: int = 1024,
        dropout: float = 0.1,
        max_len: int = 5000
    ):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers

        # 词嵌入 + 位置编码
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)

        # 堆叠多层编码器
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        # 最终层归一化
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, src_tokens: torch.Tensor, src_padding_mask: torch.Tensor = None):
        """
        Args:
            src_tokens: [batch, seq_len] 源语言单词索引
            src_padding_mask: [batch, seq_len] padding掩码
        Returns:
            memory: [batch, seq_len, d_model] 编码器最终输出，供解码器交叉注意力使用
            attn_weights_list: 每层注意力权重，用于可视化
        """
        # 1. 词嵌入 + 缩放 + 位置编码
        x = self.embedding(src_tokens) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # 2. 逐层通过编码器
        attn_weights_list = []
        for layer in self.layers:
            x, attn_w = layer(x, src_padding_mask)
            attn_weights_list.append(attn_w)

        # 3. 最终归一化
        x = self.final_norm(x)

        return x, None

解码器：多头注意力 + 编码器-解码器注意力 + 基于位置的前馈网络

In [ ]:
# ==================== 单层解码器块 ====================
class TransformerDecoderLayer(nn.Module):
    """
    Transformer 解码器层，3个子层 + 3个 AddNorm
    1. 掩码自注意力（因果掩码，禁止看到未来token）
    2. 编码器-解码器交叉注意力（KV来自编码器输出）
    3. 逐位置前馈网络
    """
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        # 1. 掩码自注意力
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.norm1 = AddNorm(d_model, dropout)

        # 2. 编码器-解码器交叉注意力
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.norm2 = AddNorm(d_model, dropout)

        # 3. 前馈网络
        self.ffn = PositionWiseFFN(d_model, dim_feedforward, dropout)
        self.norm3 = AddNorm(d_model, dropout)

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_padding_mask: torch.Tensor = None,
        tgt_padding_mask: torch.Tensor = None
    ):
        """
        Args:
            tgt: [batch, tgt_len, d_model] 解码器输入特征
            memory: [batch, src_len, d_model] 编码器输出（交叉注意力的KV来源）
            tgt_mask: [tgt_len, tgt_len] 因果掩码，防止看到未来token
            memory_padding_mask: [batch, src_len] 源语言padding掩码
            tgt_padding_mask: [batch, tgt_len] 目标语言padding掩码
        Returns:
            out: [batch, tgt_len, d_model] 层输出
            self_attn_weights: 自注意力权重
            cross_attn_weights: 交叉注意力权重
        """
        # 1. 掩码自注意力 + 残差归一化
        self_attn_out, self_attn_weights = self.self_attn(
            query=tgt, key=tgt, value=tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_padding_mask,
            need_weights=True
        )
        tgt = self.norm1(tgt, self_attn_out)

        # 2. 交叉注意力 + 残差归一化
        cross_attn_out, cross_attn_weights = self.cross_attn(
            query=tgt, key=memory, value=memory,
            key_padding_mask=memory_padding_mask,
            need_weights=True
        )
        tgt = self.norm2(tgt, cross_attn_out)

        # 3. 前馈网络 + 残差归一化
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt, ffn_out)

        return tgt, self_attn_weights, cross_attn_weights

# ==================== Transformer 解码器 ====================
class TransformerDecoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 3,
        dim_feedforward: int = 1024,
        dropout: float = 0.1,
        max_len: int = 5000,
        tie_weights: bool = True  # 权重绑定：词嵌入与输出层共享权重，工业标准
    ):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.vocab_size = vocab_size

        # 词嵌入 + 位置编码
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)

        # 堆叠多层解码器
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        # 最终层归一化
        self.final_norm = nn.LayerNorm(d_model)

        # 输出投影层，映射到词表维度
        self.output_proj = nn.Linear(d_model, vocab_size)

        # 权重绑定：嵌入层和输出层共享权重，减少参数量，提升翻译效果
        if tie_weights:
            self.output_proj.weight = self.embedding.weight

    def _generate_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """生成下三角因果掩码：True表示该位置被屏蔽，无法被注意力看到"""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        return mask

    def init_state(self, enc_outputs, enc_hidden, enc_valid_len):
        batch, src_len, _ = enc_outputs.shape
        device = enc_outputs.device
        memory_padding_mask = torch.arange(src_len, device=device)[None, :] >= enc_valid_len[:, None]
        return (enc_outputs, memory_padding_mask)

    def forward(self, tgt_tokens, state):
        """
        训练模式前向传播：整句并行计算
        Args:
            tgt_tokens: [batch, tgt_len] 目标语言token索引（带起始符、移位）
            memory: [batch, src_len, d_model] 编码器输出
            tgt_padding_mask: [batch, tgt_len] 目标语言padding掩码
            memory_padding_mask: [batch, src_len] 源语言padding掩码
        Returns:
            logits: [batch, tgt_len, vocab_size] 每个位置的词预测分数
            attn_weights: (self_attn_list, cross_attn_list) 每层注意力权重
        """

        memory, memory_padding_mask = state
        batch, tgt_len = tgt_tokens.shape
        device = tgt_tokens.device

        # 1. 词嵌入 + 缩放 + 位置编码
        x = self.embedding(tgt_tokens) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # 2. 生成因果掩码
        causal_mask = self._generate_causal_mask(tgt_len, device)

        # 3. 逐层通过解码器
        self_attn_list = []
        cross_attn_list = []
        for layer in self.layers:
            x, self_attn_w, cross_attn_w = layer(
                x, memory,
                tgt_mask=causal_mask,
                memory_padding_mask=memory_padding_mask
            )
            self_attn_list.append(self_attn_w)
            cross_attn_list.append(cross_attn_w)

        # 4. 最终归一化 + 投影到词表
        x = self.final_norm(x)
        logits = self.output_proj(x)
        self.attention_weights = cross_attn_list
        return logits, state


class EncoderDecoder(nn.Module):
    "整体封装"

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src_tokens, tgt_tokens, src_valid_len):
        batch, src_len = src_tokens.shape
        device = src_tokens.device
        src_padding_mask = torch.arange(src_len, device=device)[None, :] >= src_valid_len[:, None]
        enc_out, _ = self.encoder(src_tokens, src_padding_mask)
        dec_state = self.decoder.init_state(enc_out, None, src_valid_len)
        Y_hat, _ = self.decoder(tgt_tokens, dec_state)
        return Y_hat, None

In [ ]:
def sequence_mask(x: torch.Tensor, valid_lens: torch.Tensor, mask_value: float = -1e6) -> torch.Tensor:
    """
    对二维张量每行做尾部掩码：索引 >= valid_len 全部填充 mask_value
    Args:
        x: shape [N, L]，N行序列，每行长度L
        valid_lens: shape [N]，每行对应的有效长度
        mask_value: 掩码填充极小值
    Returns:
        掩码后的同shape张量
    """
    n_seq, seq_len = x.shape
    # 生成位置索引 0,1,...,seq_len-1
    pos = torch.arange(seq_len, device=x.device, dtype=torch.long)
    # 广播构造掩码：[N, L]，True代表需要掩码（超出有效长度）
    mask = pos.unsqueeze(0) >= valid_lens.unsqueeze(1)
    # 原地替换掩码区域
    x = x.masked_fill(mask, mask_value)
    return x

def grad_clipping(net: nn.Module, theta: float) -> None:
    """梯度裁剪，防止RNN梯度爆炸"""
    params = [p for p in net.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for p in params:
            p.grad.data.mul_(theta / norm)

class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    "掩码交叉熵损失"
    
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label, dtype=torch.float32)
        weights = sequence_mask(weights, valid_len, mask_value=0.0)
        self.reduction = "none"
        raw_loss = super().forward(pred.permute(0, 2, 1), label)
        return (raw_loss * weights).mean(dim=1)

In [ ]:
# ---------- 掩码交叉熵损失 ----------
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):

    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label, dtype=torch.float32)
        weights = sequence_mask(weights, valid_len, mask_value=0.0)
        self.reduction = "none"
        raw_loss = super().forward(pred.permute(0, 2, 1), label)
        return (raw_loss * weights).mean(dim=1)


# ---------- 训练函数 ----------
def train_seq2seq(net: nn.Module, train_iter, lr: float, num_epochs: int,
                  tgt_vocab: Vocab, device: torch.device):
    def _xavier_init(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    net.apply(_xavier_init)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    criterion = MaskedSoftmaxCELoss()
    net.train()
    loss_history = []
    
    # 混合精度核心
    scaler = torch.cuda.amp.GradScaler()
    torch.backends.cudnn.benchmark = True # 固定输入尺寸加速卷积/矩阵乘

    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        total_tokens = 0
        for batch in train_iter:
            X, Y, X_len, Y_len = batch
            bos = torch.full((Y.shape[0], 1), tgt_vocab["<bos>"], dtype=torch.long, device=device)
            dec_input = torch.cat([bos, Y[:, :-1]], dim=1)
            
            optimizer.zero_grad()
            # 前向自动FP16
            with torch.cuda.amp.autocast(dtype=torch.float16):
                Y_hat, _ = net(X, dec_input, X_len)
                loss = criterion(Y_hat, Y, Y_len)
            
            # 反向缩放梯度防止下溢
            scaler.scale(loss.sum()).backward()
            grad_clipping(net, theta=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.sum().item()
            total_tokens += Y_len.sum().item()

        avg_loss = total_loss / total_tokens
        loss_history.append(avg_loss)
        if epoch % 10 == 0:
            print(f"epoch {epoch:3d} | loss {avg_loss:.4f} | ppl {math.exp(avg_loss):.3f}")

    plt.figure(figsize=(6, 4))
    plt.plot(range(1, num_epochs+1), loss_history, linewidth=1.5)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(alpha=0.3)
    plt.title("Transformer Translation Training Loss")
    plt.show()

# ---------- 贪心预测函数 ----------
def predict_seq2seq(net: nn.Module, src_sentence: str, src_vocab: Vocab,
                    tgt_vocab: Vocab, num_steps: int, device: torch.device,
                    save_attention: bool = False):
    
    net.eval()

    # 源句子预处理
    src_tokens = src_vocab[src_sentence.lower().split()] + [src_vocab["<eos>"]]
    src_tokens, real_len = truncate_pad(src_tokens, num_steps, src_vocab["<pad>"])
    src_valid_len = torch.tensor([real_len], device=device) # 有效长度用截断前原始长度
    enc_X = torch.tensor(src_tokens, dtype=torch.long, device=device).unsqueeze(0)
    
    # 编码
    enc_out, enc_hid = net.encoder(enc_X)
    dec_state = net.decoder.init_state(enc_out, enc_hid, src_valid_len)
    # 初始解码输入
    dec_X = torch.tensor([[tgt_vocab["<bos>"]]], dtype=torch.long, device=device)

    output_ids = []
    attn_seq = []
    with torch.no_grad():
        for _ in range(num_steps):
            Y, dec_state = net.decoder(dec_X, dec_state)
            dec_X = Y.argmax(dim=2)
            pred_id = dec_X.item()
            if save_attention:
                attn_seq.append(net.decoder.attention_weights)
            if pred_id == tgt_vocab["<eos>"]:
                break
            output_ids.append(pred_id)

    translation = " ".join(tgt_vocab.to_tokens(output_ids))
    return translation, attn_seq


# ---------- BLEU计算 ----------
def bleu(pred_seq: str, label_seq: str, k: int = 2) -> float:
    pred_tokens = pred_seq.split()
    label_tokens = label_seq.split()
    len_pred, len_label = len(pred_tokens), len(label_tokens)
    # 简洁惩罚
    score = math.exp(min(0.0, 1.0 - len_label / max(len_pred, 1)))
    for n in range(1, k + 1):
        label_ngram = Counter()
        for i in range(len_label - n + 1):
            label_ngram[tuple(label_tokens[i:i+n])] += 1
        pred_ngram = Counter()
        matches = 0
        for i in range(len_pred - n + 1):
            gram = tuple(pred_tokens[i:i+n])
            pred_ngram[gram] += 1
        for gram in pred_ngram:
            matches += min(pred_ngram[gram], label_ngram.get(gram, 0))
        precision = matches / max(1, len_pred - n + 1)
        score *= math.pow(precision, 1.0 / k)
    return score

In [ ]:
num_hiddens = 32 # 模型特征维度
num_layers = 2   # 编码器、解码器各自堆叠的层数
dropout = 0.1 
batch_size = 64 
num_steps = 10   # 序列最大长度
lr = 0.005
num_epochs = 200

ffn_num_input = 32
ffn_num_hiddens = 64 # FFN 中间隐藏层维度
num_heads = 4        # 多头注意力头数
key_size, query_size, value_size = 32, 32, 32
norm_shape = [32]    # LayerNorm 归一化维度

def get_device(i: int = 0) -> torch.device:
    if torch.cuda.is_available() and torch.cuda.device_count() > i:
        return torch.device(f"cuda:{i}")
    return torch.device("cpu")

device = get_device()

train_iter, src_vocab, tgt_vocab = load_data_nmt(
    batch_size = batch_size,
    max_len = num_steps,
    src_file = "train_src.txt",
    tgt_file = "train_tgt.txt",
    min_freq = 1,
    shuffle = True,
    device=device
)

# 编码器
encoder = TransformerEncoder(
    vocab_size=len(src_vocab),
    d_model=num_hiddens,
    nhead=num_heads,
    num_layers=num_layers,
    dim_feedforward=ffn_num_hiddens,
    dropout=dropout
)

# 解码器
decoder = TransformerDecoder(
    vocab_size=len(tgt_vocab),
    d_model=num_hiddens,
    nhead=num_heads,
    num_layers=num_layers,
    dim_feedforward=ffn_num_hiddens,
    dropout=dropout
)

# 整体封装
net = EncoderDecoder(encoder, decoder)

In [ ]:
"模型训练"

train_seq2seq(
    net=net,
    train_iter=train_iter,
    lr=lr,
    num_epochs=num_epochs,
    tgt_vocab=tgt_vocab,
    device=device
)

In [ ]:
"模型预测"

# 测试句子
test_engs = [
    "i love deep learning",
    "where is the train station",
    "i drink coffee every morning",
    "he is calm"
]
test_fras = [
    "j'aime l'apprentissage profond",
    "où est la gare",
    "je bois du café chaque matin",
    "il est calme"
]

print("===== 翻译结果与BLEU评估 =====")
for eng, fra in zip(test_engs, test_fras):
    translation, _ = predict_seq2seq(
        net=net,
        src_sentence=eng,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        num_steps=num_steps,
        device=device
    )
    bleu_score = bleu(translation, fra, k=2)
    print(f"English: {eng}")
    print(f"French: {fra}")
    print(f"Model: {translation}")
    print(f"BLEU-2: {bleu_score:.3f}\n")